# Wheel Strategy — Backtest, Calibration & Rule Ablation

This notebook is the *proof* layer of `quant-wheel`. It runs the walk-forward wheel-cycle
backtest and asks the only question that matters: **do the rules actually improve the odds?**

We measure three things:
1. **Performance** — equity curve, Sharpe, max drawdown, CAGR, win rate.
2. **Calibration** — are the model's predicted assignment probabilities honest?
3. **Ablation** — how much does each rule (earnings gate, regime gate, 50% profit-take) contribute?

> **Known limitation.** Yahoo provides only *current* option chains, so historical premiums are
> modelled with Black-Scholes on *realized* vol, not historical *implied* vol. The live tool uses
> real IV; the backtest approximates it. Real income is usually a bit higher than modelled, but the
> *relative* comparison the ablation makes is unaffected. Historical earnings dates are also
> synthesized quarterly when the free feed lacks them (see `core/data/earnings.py`).

In [ ]:
from datetime import date
import pandas as pd
import matplotlib.pyplot as plt

from quantwheel.services.backtest_service import run_backtest

TICKERS = ["F", "SOFI", "NIO", "AAL"]
res = run_backtest(TICKERS, date(2024, 1, 1), date(2026, 6, 1))
print(f"{res.n_trades} trades across {res.tickers}")

## 1. Equity curve

In [ ]:
eq = pd.Series(
    {pd.Timestamp(d): v for d, v in res.equity_curve}
).sort_index()

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(eq.index, eq.values, lw=1.6)
ax.axhline(1.0, color="grey", ls="--", lw=0.8)
ax.set_title("Equal-weight wheel — normalized equity")
ax.set_ylabel("equity (start = 1.0)")
ax.grid(alpha=0.3)
plt.show()

## 2. Headline metrics

In [ ]:
pd.Series(res.metrics).to_frame("baseline").style.format("{:.4f}")

## 3. Rule ablation

Each bar is the strategy with one rule **turned off**. If a rule earns its place, removing it
should *lower* the Sharpe ratio (or blow out the drawdown).

In [ ]:
rows = {"baseline (all on)": res.metrics}
rows.update(res.ablation)
tbl = pd.DataFrame(rows).T[["sharpe", "cagr", "max_drawdown", "win_rate"]]
display(tbl.style.format("{:.3f}"))

fig, ax = plt.subplots(figsize=(9, 4))
tbl["sharpe"].plot.bar(ax=ax, color=["#2c7fb8"] + ["#d95f0e"] * len(res.ablation))
ax.set_title("Sharpe by variant (baseline vs each rule removed)")
ax.set_ylabel("Sharpe")
ax.grid(alpha=0.3, axis="y")
plt.tight_layout(); plt.show()

## 4. Calibration (are the probabilities honest?)

Predicted assignment probability vs realized frequency. Points on the diagonal = well calibrated;
below the diagonal = the model is *conservative* (predicts more assignments than happen).

In [ ]:
if res.calibration:
    pred, real = zip(*res.calibration)
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.plot([0, 1], [0, 1], ls="--", color="grey", label="perfect")
    ax.scatter(pred, real, s=60, zorder=3)
    ax.set_xlabel("predicted P(assign)"); ax.set_ylabel("realized frequency")
    ax.set_title("Assignment calibration"); ax.legend(); ax.grid(alpha=0.3)
    plt.show()
else:
    print("No expiry outcomes to calibrate (try a longer window / more tickers).")

## Conclusions

Fill in after running, e.g.:
- **Profit-take is essential** — removing it flips the Sharpe negative.
- **The regime gate** trades some raw return for a much smaller drawdown.
- **The earnings gate** adds risk-adjusted return by sitting out event weeks.
- The model is **conservative** on assignment (good — it errs toward caution).

Next: plug in a real implied-vol source (Tradier sandbox) to remove the realized-vol approximation.